# riffle — Three-way showcase: Base / SFT / SFT+GRPO

This notebook is the **exploration + figure-generation companion** to the final project report.

It does three things:

1. **Loads all three checkpoints** (base `Qwen3-1.7B`, SFT, SFT+GRPO) from Google Drive.
2. **Generates side-by-side audio** for the same `(style, structure)` prompts so we can listen to how the output changes through the pipeline.
3. **Renders all of the diagrams** used in the slides and written report. Each diagram cell saves a PNG into `figures/` for drop-in to Google Slides / LaTeX.

> Read [`[PLAN] RLVR for Chord Progressions/PROGRESS.md`](../%5BPLAN%5D%20RLVR%20for%20Chord%20Progressions/PROGRESS.md) for the full engineering trail behind the numbers shown here.

---
## 0 · Colab setup
Installs (Unsloth not needed at inference time — merged 16-bit checkpoints load with vanilla HF), FluidSynth for MIDI→WAV, then mounts Drive.

In [ ]:
!apt-get -qq install -y fluidsynth > /dev/null
!pip install -q transformers accelerate music21 midiutil midi2audio pydub matplotlib

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, sys
# Cache HF downloads on Drive so Colab restarts don't re-download base weights
os.environ['HF_HOME'] = '/content/drive/MyDrive/hf_cache'

# Clone the repo so we can import baseline_chord_gen / chord_rewards
REPO_DIR = '/content/_info5940-final'
if not os.path.isdir(REPO_DIR):
    !git clone https://github.com/your-fork/_info5940-final.git {REPO_DIR}
    # If running from your own fork, replace the URL above (or simply !cp -r the repo from Drive).
sys.path.insert(0, REPO_DIR)
%cd $REPO_DIR
os.makedirs('figures', exist_ok=True)

### Checkpoint paths
Edit these to point at the merged 16-bit checkpoint directories on Drive.

In [ ]:
MODEL_PATHS = {
    'base':     'unsloth/Qwen3-1.7B',  # HF hub ID, no fine-tuning
    'sft':      '/content/drive/MyDrive/riffle/sft-merged',
    'sft+grpo': '/content/drive/MyDrive/riffle/grpo-merged',
}
for k, p in MODEL_PATHS.items():
    print(f'  {k:9s} → {p}')

---
## 1 · Diagrams
Run these cells first — they don't need the models loaded and they all save into `figures/`.

These are the figures intended for the slides + report:

| Figure | File | Used in |
|---|---|---|
| Pipeline (Base → SFT → GRPO) | `figures/01_pipeline.png` | slide 2, report §3 |
| Sectional prompt/output format | `figures/02_task_format.png` | slide 3, report §3 |
| Reward decomposition | `figures/03_reward.png` | slide 4, report §4 |
| GRPO group-relative advantage | `figures/04_grpo.png` | slide 5, report §4 |
| Three-way eval bars | `figures/05_eval_bars.png` | slide 7, report §5 |
| Gaming rate trajectory | `figures/06_gaming.png` | slide 8, report §5 |
| Dataset Jaccard distribution | `figures/07_jaccard.png` | report §4 (limitation) |

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch
import matplotlib.patches as mpatches

plt.rcParams.update({'figure.dpi': 140, 'savefig.dpi': 200, 'font.family': 'DejaVu Sans'})

def _box(ax, xy, w, h, text, fc='#eef3fb', ec='#3d6bb5', fs=10, weight='normal'):
    x, y = xy
    ax.add_patch(FancyBboxPatch((x, y), w, h, boxstyle='round,pad=0.02',
                                 linewidth=1.4, edgecolor=ec, facecolor=fc))
    ax.text(x + w/2, y + h/2, text, ha='center', va='center',
            fontsize=fs, fontweight=weight, wrap=True)

def _arrow(ax, p0, p1, label=None, color='#444'):
    ax.add_patch(FancyArrowPatch(p0, p1, arrowstyle='->', mutation_scale=18,
                                  linewidth=1.4, color=color))
    if label:
        mx, my = (p0[0]+p1[0])/2, (p0[1]+p1[1])/2
        ax.text(mx, my + 0.05, label, ha='center', fontsize=9, color=color)

### 1.1 · Training pipeline

In [ ]:
fig, ax = plt.subplots(figsize=(11, 3.2))
ax.set_xlim(0, 11); ax.set_ylim(0, 3.2); ax.axis('off')

_box(ax, (0.2, 1.0), 2.0, 1.2,
     'Qwen3-1.7B\n(base, zero-shot)', fc='#f3f0ea', ec='#a07a3a', weight='bold')
_box(ax, (3.3, 1.0), 2.0, 1.2,
     'SFT\nChordonomicon\n~33k songs', fc='#eaf3eb', ec='#4f8a55', weight='bold')
_box(ax, (6.4, 1.0), 2.0, 1.2,
     'SFT + GRPO\nverifiable reward', fc='#f0eafa', ec='#6a4faa', weight='bold')
_box(ax, (9.0, 1.0), 1.8, 1.2,
     'Three-way\neval grid\n(style×structure)', fc='#fbeaea', ec='#a04545')

_arrow(ax, (2.25, 1.6), (3.25, 1.6), 'supervised\nfine-tune')
_arrow(ax, (5.35, 1.6), (6.35, 1.6), 'RLVR\n(GRPO)')
_arrow(ax, (8.45, 1.6), (8.95, 1.6))

ax.text(5.5, 2.7, 'Pipeline: Base → SFT → SFT+GRPO', ha='center', fontsize=13, fontweight='bold')
ax.text(5.5, 0.35,
        'LoRA r=32 α=64 on attn+MLP · merged 16-bit checkpoints · T4 16GB · TRL + Unsloth',
        ha='center', fontsize=9, color='#555')

plt.tight_layout(); plt.savefig('figures/01_pipeline.png', bbox_inches='tight'); plt.show()

### 1.2 · Task format (sectional prompt → output)

In [ ]:
fig, ax = plt.subplots(figsize=(11, 3.6))
ax.set_xlim(0, 11); ax.set_ylim(0, 3.6); ax.axis('off')

prompt_txt = ('PROMPT\n\n'
              'Style: jazz\n'
              'Structure: intro, verse, chorus,\n'
              '           verse, chorus, outro\n'
              'Rules: tag every section,\n'
              '       2–16 chords each, in order')
output_txt = ('OUTPUT\n\n'
              '<intro_1>  Cmaj7  G   Am   F\n'
              '<verse_1>  Am7  Dm7  G7  Cmaj7\n'
              '<chorus_1> F  Fm  Cmaj7  E7  Am7\n'
              '<verse_2>  Am7  Dm7  G7  Cmaj7\n'
              '<chorus_2> F  Fm  Cmaj7  E7  Am7\n'
              '<outro_1>  Am7  Fmaj7  Cmaj7  G')

_box(ax, (0.4, 0.4), 4.5, 2.8, prompt_txt, fc='#eef3fb', ec='#3d6bb5', fs=10)
_box(ax, (6.1, 0.4), 4.5, 2.8, output_txt, fc='#f4f0e6', ec='#a07a3a', fs=10)
_arrow(ax, (4.95, 1.8), (6.05, 1.8), 'LM')
ax.text(5.5, 3.3, 'Sectional chord-progression task', ha='center', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.savefig('figures/02_task_format.png', bbox_inches='tight'); plt.show()

### 1.3 · Reward decomposition
The current reward is `0.35·presence + 0.35·order + 0.30·diversity`. Compliance dominates so the model can't trade structure for diversity; diversity gives GRPO non-zero advantages once SFT saturates compliance.

Diversity uses **mean** pairwise set-Jaccard between distinct section types — calibrated against the dataset mean Jaccard of ≈0.62. *(Limitation: set-Jaccard ignores order — `[C G Am F]` vs `[F Am G C]` looks identical.)*

In [ ]:
fig, ax = plt.subplots(figsize=(11, 3.6))
ax.set_xlim(0, 11); ax.set_ylim(0, 3.6); ax.axis('off')

_box(ax, (0.4, 1.2), 3.0, 1.4,
     'presence  (0.35)\n\n|requested ∩ produced|\n────────────────\n|requested|',
     fc='#eaf3eb', ec='#4f8a55')
_box(ax, (4.0, 1.2), 3.0, 1.4,
     'order  (0.35)\n\nLCS(req_seq, out_seq)\n────────────────\n|req_seq|',
     fc='#eaf3eb', ec='#4f8a55')
_box(ax, (7.6, 1.2), 3.0, 1.4,
     'diversity  (0.30)\n\nclip((1 − mean_jaccard)/0.4,\n        0, 1)',
     fc='#f0eafa', ec='#6a4faa')

ax.text(5.5, 3.2, 'Verifiable reward = 0.35·presence + 0.35·order + 0.30·diversity',
        ha='center', fontsize=13, fontweight='bold')
ax.text(5.5, 0.5,
        'Parseability gate: no valid section tags → reward = 0.0.   Valid: reward > 0.5 and no errors.',
        ha='center', fontsize=9, color='#555')
plt.tight_layout(); plt.savefig('figures/03_reward.png', bbox_inches='tight'); plt.show()

### 1.4 · GRPO — group-relative advantage
For each prompt we sample G rollouts, score each one with the verifiable reward, then z-score within the group. The policy gradient pushes up rollouts above the group mean and pushes down rollouts below it — **no value network needed**.

In [ ]:
import numpy as np
fig, axes = plt.subplots(1, 2, figsize=(11, 3.2))

ax = axes[0]
rewards = np.array([0.92, 0.74, 0.55, 0.40])
groups = ['rollout 1', 'rollout 2', 'rollout 3', 'rollout 4']
ax.bar(groups, rewards, color='#3d6bb5')
ax.axhline(rewards.mean(), color='#a04545', linestyle='--', label=f'mean = {rewards.mean():.2f}')
ax.set_ylim(0, 1.0); ax.set_ylabel('reward'); ax.set_title('1.  Sample G rollouts per prompt')
ax.legend(loc='upper right', fontsize=9)

ax = axes[1]
adv = (rewards - rewards.mean()) / (rewards.std() + 1e-8)
colors = ['#4f8a55' if a > 0 else '#a04545' for a in adv]
ax.bar(groups, adv, color=colors)
ax.axhline(0, color='#444', linewidth=0.8)
ax.set_ylabel('advantage (z-scored)'); ax.set_title('2.  Group-relative advantage → policy gradient')

plt.tight_layout(); plt.savefig('figures/04_grpo.png', bbox_inches='tight'); plt.show()

### 1.5 · Three-way eval results
Round-1 numbers from `PROGRESS.md`. Counterintuitive finding: SFT *raises* gaming rate (34%→88%) by perfecting the format and exposing the pretraining loop-bias. GRPO barely moves it because Jaccard misreads diatonic sharing as gaming — see report §6 (Limitations).

In [ ]:
import numpy as np
conds   = ['base', 'SFT', 'SFT+GRPO']
presence = [0.86, 1.00, 1.00]
order    = [0.90, 1.00, 1.00]
passrt   = [0.16, 0.95, 0.95]

x = np.arange(len(conds)); w = 0.26
fig, ax = plt.subplots(figsize=(8.5, 4.0))
ax.bar(x - w, presence, w, label='presence', color='#4f8a55')
ax.bar(x,     order,    w, label='order',    color='#3d6bb5')
ax.bar(x + w, passrt,   w, label='pass rate (reward>0.5)', color='#a07a3a')
ax.set_xticks(x); ax.set_xticklabels(conds); ax.set_ylim(0, 1.05)
ax.set_ylabel('score'); ax.set_title('Three-way eval — structural compliance')
ax.legend(loc='lower right', fontsize=9)
for i, vals in enumerate(zip(presence, order, passrt)):
    for j, v in enumerate(vals):
        ax.text(x[i] + (j-1)*w, v + 0.02, f'{v:.2f}', ha='center', fontsize=8)
plt.tight_layout(); plt.savefig('figures/05_eval_bars.png', bbox_inches='tight'); plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.6))
vals = [0.80, 0.88, 0.86]
ax.plot(conds, vals, marker='o', linewidth=2, color='#a04545')
for c, v in zip(conds, vals):
    ax.text(c, v + 0.02, f'{int(v*100)}%', ha='center', fontsize=10)
ax.set_ylim(0, 1.0); ax.set_ylabel('gaming rate (mean_jaccard > 0.9)')
ax.set_title('Gaming rate gets WORSE through training (Jaccard limitation)')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout(); plt.savefig('figures/06_gaming.png', bbox_inches='tight'); plt.show()

### 1.6 · Dataset mean-Jaccard distribution
Calibrates the diversity reward. Run on the val split — see `scripts/analyze_section_jaccard_dataset.py`.

In [ ]:
import json, numpy as np
from chord_rewards import parse_sectional_progression

def mean_jaccard_of_row(parsed):
    sets = {}
    for name, _, chords in parsed['sections']:
        sets.setdefault(name, set()).update(chords)
    keys = list(sets.keys())
    if len(keys) < 2: return None
    js = []
    for i in range(len(keys)):
        for j in range(i+1, len(keys)):
            a, b = sets[keys[i]], sets[keys[j]]
            js.append(len(a & b) / max(1, len(a | b)))
    return float(np.mean(js))

vals = []
with open('data/chordonomicon_val.jsonl') as f:
    for line in f:
        r = json.loads(line)
        parsed = parse_sectional_progression(r.get('completion', r.get('chords','')))
        if parsed:
            mj = mean_jaccard_of_row(parsed)
            if mj is not None: vals.append(mj)

fig, ax = plt.subplots(figsize=(7.5, 3.6))
ax.hist(vals, bins=40, color='#3d6bb5', alpha=0.85)
ax.axvline(np.mean(vals), color='#a04545', linestyle='--',
           label=f'dataset mean = {np.mean(vals):.2f}')
ax.axvline(0.4, color='#4f8a55', linestyle=':',
           label='diversity-reward calibration = 0.4')
ax.set_xlabel('mean pairwise Jaccard between distinct section types')
ax.set_ylabel('# val songs'); ax.set_title('Chordonomicon - section diversity distribution')
ax.legend(fontsize=9)
plt.tight_layout(); plt.savefig('figures/07_jaccard.png', bbox_inches='tight'); plt.show()


---
## 2 · Three-way audio showcase

Pick a few `(style, structure)` prompts and render audio from each checkpoint. We unload between checkpoints so the T4 doesn't OOM.

**Listen for**: base often loops the same 4 chords across every tag (audible "that's the same loop again"); SFT outputs musically richer sections but may still copy across same-type sections; SFT+GRPO should keep that and improve presence/order — though as the eval shows, the audible diversity gap between SFT and GRPO is small.

In [ ]:
import gc, torch, baseline_chord_gen
from baseline_chord_gen import generate_sectional
from IPython.display import display, Markdown, Audio

PROMPTS = [
    ('jazz', ['intro','verse','chorus','verse','chorus','outro']),
    ('pop',  ['verse','chorus','verse','chorus','bridge','chorus']),
    ('rock', ['intro','verse','chorus','bridge','chorus','outro']),
]

def _reset_model_cache():
    baseline_chord_gen._model = None
    baseline_chord_gen._tokenizer = None
    gc.collect(); torch.cuda.empty_cache()

def render_for(label, model_id, style, sections, idx):
    _reset_model_cache()
    midi = f'/content/out_{idx}_{label}.mid'
    mp3  = f'/content/out_{idx}_{label}.mp3'
    res  = generate_sectional(
        style=style, sections=sections,
        output_midi_path=midi, output_mp3_path=mp3,
        model_id=model_id, temperature=0.7, max_new_tokens=768,
        quiet=True,
    )
    return res

In [ ]:
results = []  # (idx, style, label, GenerationResultSectional)
for idx, (style, sections) in enumerate(PROMPTS):
    display(Markdown(f'### Prompt {idx+1} — *{style}* · `{"-".join(sections)}`'))
    for label, model_id in MODEL_PATHS.items():
        display(Markdown(f'**{label}**'))
        res = render_for(label, model_id, style, sections, idx)
        results.append((idx, style, label, res))
        # parsed sections
        if res.parsed_sections:
            lines = [f'  `{n}_{i}` → {chs}' for (n, i, chs) in res.parsed_sections]
            display(Markdown('\n'.join(lines)))
        else:
            display(Markdown('_(unparsed output)_'))
        display(Markdown(f'reward = **{res.reward:.3f}**   ·   parseable = {res.parsed_sections is not None}   ·   RLVR-pass = {res.valid}   ·   gamed = {res.breakdown.get("gamed", "?")}'))
        if res.mp3_path and os.path.exists(res.mp3_path):
            display(Audio(res.mp3_path))
        else:
            display(Markdown('_(no audio — invalid output)_'))

---
## 3 · Saved figures
Everything under `figures/` is now ready to drop into the slide deck or LaTeX report.

In [ ]:
import os
for f in sorted(os.listdir('figures')):
    print('  figures/' + f)